In [2]:
import torch
import numpy as np
import math
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------------------------------------------------
# Rastrigin‑like loss with coupling
# ----------------------------------------------------------------------
def loss_function(x):
    A = 10
    return (torch.sum(x**2 - A * torch.cos(2. * torch.pi * x))
            + torch.sum(torch.stack([(x[ii+1] - x[ii] - ii/5.)**2
                                     for ii in range(len(x)-1)])))/len(x.detach().cpu().numpy())

# ----------------------------------------------------------------------
# NDDA index as defined in the paper (monitoring)
# ----------------------------------------------------------------------
def compute_NDDA(theta, sigma=0.05, alpha=1.3):
    """Return NDDA = #{i | I₁,i ≥ 0 and I₂,i ≥ 0} / d."""
    theta_ref = theta.clone().detach().to(device).requires_grad_(True)
    epsilon = sigma * torch.randn_like(theta_ref)

    # I₁
    theta_plus_eps = theta_ref + epsilon
    loss1 = loss_function(theta_plus_eps)
    loss1.backward()
    I1 = (epsilon * theta_ref.grad).detach()
    theta_ref.grad.zero_()

    # ε' with symmetric amplification α and sign flip where I₁ ≥ 0
    epsilon_prime = alpha * epsilon.clone()
    epsilon_prime[I1 >= 0] *= -1.0

    # I₂
    theta_plus_eps_prime = theta_ref + epsilon_prime
    loss2 = loss_function(theta_plus_eps_prime)
    loss2.backward()
    I2 = (epsilon_prime * theta_ref.grad).detach()
    theta_ref.grad.zero_()

    ndda = ((I1 >= 0) & (I2 >= 0)).float().mean().item()
    return ndda

# ----------------------------------------------------------------------
# Hessian diagnostics utility
# ----------------------------------------------------------------------
def hessian_diagnostics(theta):
    """
    Returns (min_eig, has_negative, max_grad_norm, is_grad_zero)
    for the current point theta.
    """
    theta = theta.clone().detach().to(device).requires_grad_(True)
    y = loss_function(theta)
    grad = torch.autograd.grad(y, theta, create_graph=True)[0]

    dim = theta.shape[0]
    H = torch.zeros(dim, dim, device=device)
    for i in range(dim):
        grad2 = torch.autograd.grad(grad[i], theta, create_graph=True, retain_graph=True)[0]
        H[i] = grad2.detach()

    eigenvalues, _ = torch.linalg.eig(H)
    eigenvalues = eigenvalues.real
    min_eig = torch.min(eigenvalues).item()
    has_neg = (eigenvalues < 0).any().item()
    max_grad_norm = torch.max(torch.abs(grad)).item()
    is_zero = torch.allclose(grad, torch.zeros_like(grad), atol=1e-8)

    return min_eig, has_neg, max_grad_norm, is_zero

# ----------------------------------------------------------------------
# CPA – Controlled Perturbation Algorithm
# ----------------------------------------------------------------------
def CPA(mu_init, sigma=0.05, alpha=1.3):
    mu = mu_init.clone().detach().to(device).requires_grad_(True)
    losses = []
    ndda_vals = []
    min_eig_vals = []
    max_grad_vals = []

    len_of_mu = len(mu)

        # 1. sample ε
    epsilon = (sigma * torch.randn_like(mu)).detach().to(device).requires_grad_(False)

        # 2. I₁
    mu_plus_eps = mu + epsilon
    energy1 = loss_function(mu_plus_eps)
    energy1.backward()
    I1 = (epsilon * mu.grad).detach()
    mu.grad.zero_()

        # 3. ε' with amplification
    epsilon_prime = alpha * epsilon.clone()
    epsilon_prime[I1 >= 0] *= -1.0

        # 4. I₂
    epsilon_prime = epsilon_prime.detach().to(device).requires_grad_(False)
    mu_plus_eps_prime = mu + epsilon_prime
    energy2 = loss_function(mu_plus_eps_prime)
    energy2.backward()
    I2 = (epsilon_prime * mu.grad).detach()
    mu.grad.zero_()

    neg1 = I1 < 0
    neg2 = I2 < 0

        # 5. Update rule
    with torch.no_grad():
        delta = torch.where(neg2, epsilon_prime,
                            torch.where(neg1, epsilon, torch.zeros_like(epsilon)))
        mu += delta

    return mu

# ----------------------------------------------------------------------
# GD and PGD (standard optimisers)
# ----------------------------------------------------------------------
def optimize_GD_then_PGD_or_CPA(params_init, escape_method = "CPA", factor_begin=0.05, factor_end=0.001, epochs=10, alpha=1.3):
    method = "GD"
    params = params_init.clone().detach().to(device).requires_grad_(True)
    losses = []
    ndda_vals = []
    min_eig_vals = []
    max_grad_vals = []

    for epoch in range(epochs):
        lrlr = factor_begin * (1. - epoch/epochs) + factor_end * epoch/epochs # diminishing LR
        energy = loss_function(params)
        losses.append(energy.item())
        

        
        if method == "GD":
            energy.backward()
            with torch.no_grad():
                params -= lrlr * params.grad
        elif method == "PGD":
                energy.backward()
                with torch.no_grad():
                    noise = torch.randn_like(params) * lrlr
                    params -= lrlr * params.grad + noise
        elif method == "CPA":
            params= CPA(params, sigma=lrlr, alpha=alpha)
        params.grad.zero_()

        # NDDA
        ndda = compute_NDDA(params, sigma=0.05, alpha=alpha)
        ndda_vals.append(ndda)


        para_temp = params.clone().detach().to(device).requires_grad_(True)
        y = loss_function(para_temp)
        grad = torch.autograd.grad(y, para_temp, create_graph=True)[0]
        max_grad_norm = torch.max(torch.abs(grad)).item()
        if max_grad_norm < 8e-3:
            temp = method
            method = escape_method
            if temp != escape_method:
                print(f"Switching to {escape_method} at epoch {epoch} due to small gradient norm: {max_grad_norm:.5e}")
        else:
            temp = method
            method="GD"
            if temp != "GD":
                print(f"Switching back to GD at epoch {epoch} due to large gradient norm: {max_grad_norm:.5e}")    


        # Hessian check every 100 epochs
        if epoch % 100 == 0:
            min_eig, has_neg, max_g, is_zero = hessian_diagnostics(params)
            min_eig_vals.append(min_eig)
            max_grad_vals.append(max_g)
            print(f"epoch {epoch:4d}  {method.upper()}  loss={losses[-1]:.6f}  NDDA={ndda:.3f}  "
                  f"min eig={min_eig:.4f}  negative eig={has_neg}  max grad={max_g:.6f}  grad zero={is_zero}")
        else:
            if len(min_eig_vals) > 0:
                min_eig_vals.append(min_eig_vals[-1])
                max_grad_vals.append(max_grad_vals[-1])

    return params, losses, ndda_vals, min_eig_vals, max_grad_vals

# ----------------------------------------------------------------------
# Experiment setup
# ----------------------------------------------------------------------
N = 100
epoch_no = 10000
torch.manual_seed(0)

# Initial point
params0 = 1.0 + 0.1 * torch.from_numpy(np.random.normal(size=N))
for i in range(11):
    params0[i] += i
params0 = params0.float()


print("=== CPA ===")
_, cpa_losses, cpa_ndda, cpa_min_eig, cpa_max_grad = optimize_GD_then_PGD_or_CPA(
    params0, escape_method="CPA", factor_begin=0.1, factor_end=0.005, epochs=epoch_no, alpha=1.3)
print("\n=== PGD ===")
_, pgd_losses, pgd_ndda, pgd_min_eig, pgd_max_grad = optimize_GD_then_PGD_or_CPA(
    params0, escape_method="PGD", factor_begin=0.1, factor_end=0.005, epochs=epoch_no, alpha=1.3)



print("\nFinal losses:")
print(f"PGD: {pgd_losses[-1]:.6f}")
print(f"CPA: {cpa_losses[-1]:.6f}")

# ----------------------------------------------------------------------
# Plotting
# ----------------------------------------------------------------------

# Losses
plt.figure(figsize=(9,5))
plt.plot(pgd_losses[300:], label="GD+PGD")
plt.plot(cpa_losses[300:], label="GD+CPA")
plt.xlabel("Epoch")
plt.ylabel("Function Value")
plt.legend()
plt.title("Rastrigin Function – Loss")
plt.savefig("Rastrigin_Loss_Corrected_with_Hessian.png")
plt.close()

# NDDA
plt.figure(figsize=(9,5))
plt.plot(pgd_ndda[300:], label="GD+PGD")
plt.plot(cpa_ndda[300:], label="GD+CPA")
plt.xlabel("Epoch")
plt.ylabel("NDDA Index")
plt.legend()
plt.title("Rastrigin Function – NDDA Index")
plt.savefig("Rastrigin_NDDA_Corrected_with_Hessian.png")
plt.close()

# Minimum eigenvalue (only at logged epochs)
plt.figure(figsize=(9,5))
plt.plot(pgd_min_eig[300:], label="GD+PGD")
plt.plot(cpa_min_eig, label="CPA")
plt.xlabel("Logged epoch (every 100)")
plt.ylabel("Min eigenvalue of Hessian")
plt.legend()
plt.title("Rastrigin Function – Min Hessian Eigenvalue")
plt.savefig("Rastrigin_MinEig_Corrected_with_Hessian.png")
plt.close()

# Max gradient
plt.figure(figsize=(9,5))
plt.plot(pgd_max_grad[300:], label="GD+PGD")
plt.plot(cpa_max_grad[300:], label="GD+CPA")
plt.xlabel("Logged epoch (every 100)")
plt.ylabel("Max absolute gradient")
plt.legend()
plt.title("Rastrigin Function – Max Gradient")
plt.savefig("Rastrigin_MaxGrad_Corrected_with_Hessian.png")
plt.close()

# Print some final stats
print("\nFinal Hessian statistics:")
print(f"GD+PGD: min eig = {pgd_min_eig[-1]:.4f}  max grad = {pgd_max_grad[-1]:.6f}")
print(f"GD+CPA: min eig = {cpa_min_eig[-1]:.4f}  max grad = {cpa_max_grad[-1]:.6f}")

=== CPA ===
epoch    0  GD  loss=126.217545  NDDA=0.480  min eig=0.2804  negative eig=False  max grad=0.619262  grad zero=False
Switching to CPA at epoch 11 due to small gradient norm: 5.29777e-03
Switching back to GD at epoch 36 due to large gradient norm: 3.68900e-02
Switching to CPA at epoch 60 due to small gradient norm: 7.88431e-03
Switching back to GD at epoch 79 due to large gradient norm: 3.31016e-01
Switching to CPA at epoch 93 due to small gradient norm: 7.96116e-03
epoch  100  CPA  loss=124.213219  NDDA=0.990  min eig=3.2113  negative eig=False  max grad=0.007961  grad zero=False
Switching back to GD at epoch 147 due to large gradient norm: 1.01186e-01
Switching to CPA at epoch 167 due to small gradient norm: 7.40484e-03
Switching back to GD at epoch 194 due to large gradient norm: 3.94922e-02
epoch  200  GD  loss=124.080200  NDDA=0.990  min eig=-3.3608  negative eig=True  max grad=0.254537  grad zero=False
Switching to CPA at epoch 216 due to small gradient norm: 5.01782e-0

save_params, save_sgld_losses, save_nonnegative_slope_ratio = params_3, sgld_losses, sgld_negative_slope_ratio